# Myeloid Cell Annotation Refinement and Re-integration Pipeline

**Author:** r2end  
**Date:** 2026-02-10  
**Version:** 2.0

## Purpose
1. Load raw myeloid h5ad with original annotations
2. Apply refined cell type annotations (with merging)
3. Re-train scVI model for batch correction
4. Re-train scANVI model for semi-supervised annotation
5. Generate comprehensive validation visualizations
6. Save refined annotations and models

## Key Features
- **Immunoregulatory IM preserved** (Intestinal macrophages_c3)
- **CD163L1+ IM merged** (c0, c2, c4)
- **Memory optimized** (HVG training + .raw for full genes)
- **Complete validation** (dotplot + UMAPs)

**Memory:** ~30-40GB peak  
**Runtime:** ~2-3 hours (with GPU)

---
## 0. Configuration and Setup

In [ ]:
# Import libraries
import os
import sys
import gc
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse

import scanpy as sc
import scvi

print("Libraries imported successfully")
print(f"scanpy version: {sc.__version__}")
print(f"scvi-tools version: {scvi.__version__}")

In [ ]:
# ===== CONFIGURATION - UPDATE THESE PATHS =====

# Input/Output paths
INPUT_H5AD = "/home/h2048/data/py/0128/myeloid_analysis_unified/results/subcluster_unified_v2_20260128/adata_myeloid_subclustered_FINAL_v2_20260128.h5ad"
OUTPUT_DIR = Path("/home/h2048/data/py/0209/myeloid_validation_optimized")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Column names
CELLTYPE_L2_COL = 'cell_type_L2'
CELLTYPE_L3_COL = 'cell_type_L3'  # Original L3 annotation
CELLTYPE_L3_REFINED_COL = 'cell_type_L3_refined'  # New refined annotation
BATCH_KEY = 'sample'  # ⚠️ UPDATE if different

# scVI/scANVI parameters
RANDOM_SEED = 42
N_HVG = 4000
N_LATENT = 75
N_HIDDEN = 128
N_LAYERS = 2
DROPOUT_RATE = 0.1
MAX_EPOCHS_SCVI = 400
MAX_EPOCHS_SCANVI = 200

# Computational settings
N_JOBS = 4
USE_GPU = True

print("="*80)
print("CONFIGURATION")
print("="*80)
print(f"Input: {INPUT_H5AD}")
print(f"Output: {OUTPUT_DIR}")
print(f"Random seed: {RANDOM_SEED}")
print(f"N_HVG: {N_HVG}")
print(f"N_latent: {N_LATENT}")
print(f"Use GPU: {USE_GPU}")
print(f"N_jobs: {N_JOBS}")
print("="*80)

In [ ]:
# Set up scanpy and random seeds
sc.settings.verbosity = 3
sc.settings.n_jobs = N_JOBS
sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=False)

np.random.seed(RANDOM_SEED)
scvi.settings.seed = RANDOM_SEED

print("✓ Settings configured")

---
## 1. Load Data

In [ ]:
print("="*80)
print("1. LOADING DATA")
print("="*80)

adata = sc.read_h5ad(INPUT_H5AD)

print(f"\nLoaded data:")
print(f"  Cells: {adata.n_obs:,}")
print(f"  Genes: {adata.n_vars:,}")
print(f"  Layers: {list(adata.layers.keys())}")
print(f"  Batches: {adata.obs[BATCH_KEY].nunique()}")

# Verify required columns
required_cols = [CELLTYPE_L2_COL, CELLTYPE_L3_COL, BATCH_KEY]
missing_cols = [col for col in required_cols if col not in adata.obs.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print(f"\n✓ All required columns present")

In [ ]:
# Display original L3 annotation distribution
print("\nOriginal L3 annotation distribution:")
print(adata.obs[CELLTYPE_L3_COL].value_counts())

---
## 2. Standardize L3 Labels

In [ ]:
print("="*80)
print("2. STANDARDIZING L3 LABELS")
print("="*80)

def standardize_l3_labels(adata, l2_key=CELLTYPE_L2_COL, l3_key=CELLTYPE_L3_COL):
    """Standardize L3 labels to hierarchical format: L2_c#"""
    sample = str(adata.obs[l3_key].iloc[0])
    if '_c' in sample:
        print("  ✓ L3 labels already hierarchical")
        return adata
    
    print("  Standardizing L3 labels...")
    adata.obs['cell_type_L3_original'] = adata.obs[l3_key].copy()
    adata.obs['subcluster_id'] = pd.to_numeric(
        adata.obs[l3_key].astype(str), errors='coerce'
    ).astype('Int64')
    
    # Convert categorical to object to avoid assignment error
    if isinstance(adata.obs[l3_key].dtype, pd.CategoricalDtype):
        adata.obs[l3_key] = adata.obs[l3_key].astype('object')
    
    # Create hierarchical labels
    l2 = adata.obs[l2_key].astype(str)
    sid = adata.obs['subcluster_id']
    mask = l2.notna() & sid.notna()
    adata.obs.loc[mask, l3_key] = (
        l2[mask] + '_c' + sid[mask].astype('Int64').astype(str)
    )
    
    # Make categorical with proper ordering
    tmp_df = adata.obs[[l2_key, 'subcluster_id', l3_key]].dropna()
    tmp_df = tmp_df.sort_values([l2_key, 'subcluster_id'])
    ordered_levels = pd.unique(tmp_df[l3_key].astype(str)).tolist()
    
    adata.obs[l3_key] = pd.Categorical(
        adata.obs[l3_key].astype(str),
        categories=ordered_levels,
        ordered=True
    )
    
    print(f"  ✓ Standardized {len(ordered_levels)} L3 subclusters")
    return adata

adata = standardize_l3_labels(adata)

---
## 3. Apply Refined Annotation

In [ ]:
print("="*80)
print("3. APPLYING REFINED CELL TYPE ANNOTATIONS")
print("="*80)

# Define annotation mapping
CELLTYPE_REANNOTATION = {
    # Alveolar macrophages
    'Alveolar macrophages_c0': 'Resident Alveolar macrophages',
    'Alveolar macrophages_c1': 'Resident Alveolar macrophages',
    'Alveolar macrophages_c2': 'Resident Alveolar macrophages',
    'Alveolar macrophages_c3': 'Resting Alveolar macrophages',
    
    # Neutrophils / Monocytes
    'Classical monocytes_c0': 'Neutrophils',  # L2 corrected
    'Classical monocytes_c1': 'Typical Classical monocytes',
    'Classical monocytes_c2': 'Inflammatory Classical monocytes',
    
    # cDC2 (unified)
    'DC2_c0': 'Conventional cDC2',
    'DC2_c1': 'Langerhans-like cDC2',
    'DC_c0': 'Conventional cDC2',
    'DC_c1': 'Conventional cDC2',  # Merged (was Antigen-presenting)
    
    # pDC
    'pDC_c0': 'pDC',
    'pDC_c1': 'pDC',
    
    # Interstitial macrophages (from Macrophages)
    'Macrophages_c0': 'Inflammatory Interstitial macrophages',
    'Macrophages_c1': 'M2-like Interstitial macrophages',
    'Macrophages_c2': 'Atypically activated Interstitial macrophages',
    
    # Mast cells
    'Mast cells_c0': 'Mast cells',
    'Mast cells_c1': 'Mast cells',
    
    # ⭐ Interstitial macrophages (from Intestinal) - WITH IMMUNOREGULATORY PRESERVED
    'Intestinal macrophages_c0': 'CD163L1+ Interstitial macrophages',
    'Intestinal macrophages_c1': 'Low-quality Interstitial macrophages',
    'Intestinal macrophages_c2': 'CD163L1+ Interstitial macrophages',  # Merged (was Tissue-remodeling)
    'Intestinal macrophages_c3': 'Immunoregulatory Interstitial macrophages',  # ⭐ PRESERVED
    'Intestinal macrophages_c4': 'CD163L1+ Interstitial macrophages',  # Merged (was Stromal-like)
}

LOW_CONFIDENCE_CLUSTERS = [
    'Macrophages_c2',  # Atypically activated
    'Intestinal macrophages_c1',  # Low-quality
]

print("\nAnnotation mapping defined")
print(f"  Total mappings: {len(CELLTYPE_REANNOTATION)}")
print(f"  Low confidence clusters: {len(LOW_CONFIDENCE_CLUSTERS)}")

In [ ]:
# Apply annotation
adata.obs[CELLTYPE_L3_REFINED_COL] = adata.obs[CELLTYPE_L3_COL].map(CELLTYPE_REANNOTATION)

# Handle unmapped
unmapped = adata.obs[CELLTYPE_L3_COL][adata.obs[CELLTYPE_L3_REFINED_COL].isna()].unique()
if len(unmapped) > 0:
    print(f"⚠️  {len(unmapped)} unmapped clusters - keeping original labels")
    print(f"    {list(unmapped)}")
    adata.obs[CELLTYPE_L3_REFINED_COL].fillna(adata.obs[CELLTYPE_L3_COL], inplace=True)

adata.obs[CELLTYPE_L3_REFINED_COL] = adata.obs[CELLTYPE_L3_REFINED_COL].astype('category')

print(f"\n✓ Refined annotation applied: {adata.obs[CELLTYPE_L3_REFINED_COL].nunique()} unique cell types")

In [ ]:
# Display refined distribution
print("="*80)
print("REFINED CELL TYPE DISTRIBUTION")
print("="*80)

refined_counts = adata.obs[CELLTYPE_L3_REFINED_COL].value_counts()
for ct, count in refined_counts.items():
    pct = count / adata.n_obs * 100
    original_clusters = adata.obs[adata.obs[CELLTYPE_L3_REFINED_COL] == ct][CELLTYPE_L3_COL].unique()
    flag = " [⚠️ LOW CONFIDENCE]" if any(c in LOW_CONFIDENCE_CLUSTERS for c in original_clusters) else ""
    print(f"  {ct}: {count:,} cells ({pct:.1f}%){flag}")
    if len(original_clusters) > 1:
        print(f"    └─ Merged from: {', '.join(original_clusters)}")

---
## 4. Preprocessing for scVI

In [ ]:
print("="*80)
print("4. PREPROCESSING FOR scVI/scANVI")
print("="*80)

# Ensure we have counts layer
if 'counts' not in adata.layers:
    print("  ⚠️  'counts' layer not found, using .X")
    adata.layers['counts'] = adata.X.copy()
else:
    print("  ✓ 'counts' layer found")

In [ ]:
# Filter small batches
print("\nFiltering small batches...")
batch_counts = adata.obs[BATCH_KEY].value_counts()
print(f"  Batch size distribution:")
print(batch_counts.describe())

small_batches = batch_counts[batch_counts < 3].index
if len(small_batches) > 0:
    print(f"\n  Removing {len(small_batches)} small batches (< 3 cells)")
    adata = adata[~adata.obs[BATCH_KEY].isin(small_batches)].copy()
    print(f"  Remaining cells: {adata.n_obs:,}")
else:
    print("  ✓ No small batches to remove")

In [ ]:
# Basic preprocessing
print("\nBasic preprocessing...")
sc.pp.normalize_total(adata, target_sum=1e4, layer='counts', key_added='norm_factor')
sc.pp.log1p(adata, layer='counts')
adata.layers['log1p'] = adata.layers['counts'].copy()

# Set .X to log1p for HVG selection
adata.X = adata.layers['log1p'].copy()

print("  ✓ Normalization and log1p complete")

In [ ]:
# HVG selection
print(f"\nSelecting {N_HVG} highly variable genes...")
try:
    sc.pp.highly_variable_genes(
        adata, 
        n_top_genes=N_HVG, 
        batch_key=BATCH_KEY,
        subset=False
    )
    hvg_method = "batch-aware"
    print(f"  ✓ Batch-aware HVG selection")
except Exception as e:
    print(f"  ⚠️  Batch-aware HVG failed: {e}")
    print(f"  Falling back to non-batch-aware HVG")
    sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, subset=False)
    hvg_method = "non-batch-aware"

n_hvg = adata.var['highly_variable'].sum()
print(f"  ✓ Selected {n_hvg} HVGs ({hvg_method})")

In [ ]:
# ⭐ CRITICAL: Save full gene matrix to .raw (shared memory)
print("\nPreserving full gene matrix to .raw...")
adata.raw = sc.AnnData(
    X=adata.layers['counts'],  # No .copy(), shared memory
    obs=adata.obs.copy(),
    var=adata.var.copy()
)
print(f"  ✓ Full matrix preserved: {adata.raw.n_vars} genes")

In [ ]:
# Subset to HVG for training
print(f"\nSubsetting to HVG for model training...")
adata_hvg = adata[:, adata.var['highly_variable']].copy()
print(f"  ✓ Training data: {adata_hvg.n_obs:,} cells × {adata_hvg.n_vars} genes")

# Clean up
del adata
gc.collect()
print("  ✓ Memory cleaned")

---
## 5. Train scVI Model

In [ ]:
print("="*80)
print("5. TRAINING scVI MODEL")
print("="*80)

# Setup anndata for scVI
scvi.model.SCVI.setup_anndata(
    adata_hvg,
    layer='log1p',
    batch_key=BATCH_KEY
)

print("✓ AnnData setup for scVI")

In [ ]:
# Create model
print(f"\nCreating scVI model...")
print(f"  N_latent: {N_LATENT}")
print(f"  N_hidden: {N_HIDDEN}")
print(f"  N_layers: {N_LAYERS}")
print(f"  Dropout: {DROPOUT_RATE}")

vae = scvi.model.SCVI(
    adata_hvg,
    n_latent=N_LATENT,
    n_hidden=N_HIDDEN,
    n_layers=N_LAYERS,
    dropout_rate=DROPOUT_RATE,
    gene_likelihood="nb",
    use_observed_lib_size=False,
)

print("✓ scVI model created")

In [ ]:
# Train
print(f"\nTraining scVI model (max_epochs={MAX_EPOCHS_SCVI})...")
print("This will take ~30-60 minutes...")

vae.train(
    max_epochs=MAX_EPOCHS_SCVI,
    train_size=0.9,
    early_stopping=True,
)

print("\n✓ scVI training complete")

In [ ]:
# Get latent representation
print("\nGenerating scVI latent representation...")
adata_hvg.obsm['X_scvi'] = vae.get_latent_representation()
print(f"  ✓ X_scvi shape: {adata_hvg.obsm['X_scvi'].shape}")

In [ ]:
# Save scVI model
scvi_model_dir = OUTPUT_DIR / 'scvi_model'
vae.save(scvi_model_dir, overwrite=True)
print(f"\n✓ scVI model saved: {scvi_model_dir}")

---
## 6. Train scANVI Model

In [ ]:
print("="*80)
print("6. TRAINING scANVI MODEL")
print("="*80)

# Create scANVI from scVI
print("\nCreating scANVI model from trained scVI...")
lvae = scvi.model.SCANVI.from_scvi_model(
    vae,
    adata=adata_hvg,
    labels_key=CELLTYPE_L3_REFINED_COL,
    unlabeled_category='Unknown',
)

print("✓ scANVI model created")

In [ ]:
# Train
print(f"\nTraining scANVI model (max_epochs={MAX_EPOCHS_SCANVI})...")
print("This will take ~20-40 minutes...")

lvae.train(
    max_epochs=MAX_EPOCHS_SCANVI,
    train_size=0.9,
    early_stopping=True,
    
)

print("\n✓ scANVI training complete")

In [ ]:
# Get predictions and latent representation
print("\nGenerating scANVI predictions and latent...")
adata_hvg.obs['scanvi_predictions'] = lvae.predict()
adata_hvg.obsm['X_scanvi'] = lvae.get_latent_representation()
print(f"  ✓ X_scanvi shape: {adata_hvg.obsm['X_scanvi'].shape}")

# Prediction confidence
prediction_probs = lvae.predict(soft=True)
adata_hvg.obs['scanvi_confidence'] = prediction_probs.max(axis=1)
print(f"  ✓ Mean prediction confidence: {adata_hvg.obs['scanvi_confidence'].mean():.3f}")

In [ ]:
# Save scANVI model
scanvi_model_dir = OUTPUT_DIR / 'scanvi_model'
lvae.save(scanvi_model_dir, overwrite=True)
print(f"\n✓ scANVI model saved: {scanvi_model_dir}")

---
## 7. Dimensionality Reduction and Clustering

In [ ]:
print("="*80)
print("7. DIMENSIONALITY REDUCTION AND CLUSTERING")
print("="*80)

# UMAP on scVI latent
print("\nComputing UMAP on scVI latent...")
sc.pp.neighbors(adata_hvg, use_rep='X_scvi', n_neighbors=15, metric='cosine')
sc.tl.umap(adata_hvg)
print("  ✓ UMAP computed")

In [ ]:
# Leiden clustering
print("\nLeiden clustering...")
sc.tl.leiden(adata_hvg, resolution=1.0, key_added='leiden_scvi')
print(f"  ✓ {adata_hvg.obs['leiden_scvi'].nunique()} clusters")

In [ ]:
# UMAP on scANVI latent
print("\nComputing UMAP on scANVI latent...")
sc.pp.neighbors(adata_hvg, use_rep='X_scanvi', n_neighbors=15, metric='cosine', key_added='scanvi')
sc.tl.umap(adata_hvg, neighbors_key='scanvi')
adata_hvg.obsm['X_umap_scanvi'] = adata_hvg.obsm['X_umap'].copy()

# Restore original UMAP
sc.pp.neighbors(adata_hvg, use_rep='X_scvi', n_neighbors=15, metric='cosine')
sc.tl.umap(adata_hvg)

print("✓ All dimensionality reduction complete")

---
## 8. Restore Full Gene Matrix

In [ ]:
print("="*80)
print("8. RESTORING FULL GENE MATRIX")
print("="*80)

# Reload original data
print("  Loading original data...")
adata_full = sc.read_h5ad(INPUT_H5AD)

# Standardize labels
adata_full = standardize_l3_labels(adata_full)

# Apply refined annotation
adata_full.obs[CELLTYPE_L3_REFINED_COL] = adata_full.obs[CELLTYPE_L3_COL].map(CELLTYPE_REANNOTATION)
adata_full.obs[CELLTYPE_L3_REFINED_COL].fillna(adata_full.obs[CELLTYPE_L3_COL], inplace=True)
adata_full.obs[CELLTYPE_L3_REFINED_COL] = adata_full.obs[CELLTYPE_L3_REFINED_COL].astype('category')

# Filter same cells as HVG data
cell_ids = adata_hvg.obs_names
adata_full = adata_full[cell_ids, :].copy()

print(f"  ✓ Restored full matrix: {adata_full.n_obs:,} cells × {adata_full.n_vars:,} genes")

In [ ]:
# Transfer embeddings and annotations
print("\nTransferring scVI/scANVI results...")
adata_full.obsm['X_scvi'] = adata_hvg.obsm['X_scvi']
adata_full.obsm['X_scanvi'] = adata_hvg.obsm['X_scanvi']
adata_full.obsm['X_umap'] = adata_hvg.obsm['X_umap']
adata_full.obsm['X_umap_scanvi'] = adata_hvg.obsm['X_umap_scanvi']
adata_full.obs['scanvi_predictions'] = adata_hvg.obs['scanvi_predictions']
adata_full.obs['scanvi_confidence'] = adata_hvg.obs['scanvi_confidence']
adata_full.obs['leiden_scvi'] = adata_hvg.obs['leiden_scvi']

# Store HVG info
adata_full.var['highly_variable'] = False
adata_full.var.loc[adata_hvg.var_names, 'highly_variable'] = True

print("✓ All results transferred")

In [ ]:
# Add metadata
adata_full.uns['scvi_params'] = {
    'n_latent': N_LATENT,
    'n_hidden': N_HIDDEN,
    'n_layers': N_LAYERS,
    'dropout_rate': DROPOUT_RATE,
    'n_hvg': N_HVG,
    'hvg_method': hvg_method,
    'max_epochs_scvi': MAX_EPOCHS_SCVI,
    'max_epochs_scanvi': MAX_EPOCHS_SCANVI,
}

adata_full.uns['annotation_info'] = {
    'version': 'refined_v2.0',
    'date': datetime.now().strftime('%Y-%m-%d'),
    'low_confidence_clusters': LOW_CONFIDENCE_CLUSTERS,
    'merged_clusters': {
        'CD163L1+_IM': ['Intestinal macrophages_c0', 'Intestinal macrophages_c2', 'Intestinal macrophages_c4'],
        'Conventional_cDC2': ['DC2_c0', 'DC_c0', 'DC_c1'],
    },
    'preserved_clusters': {
        'Immunoregulatory_IM': 'Intestinal macrophages_c3'
    },
    'total_cell_types': adata_full.obs[CELLTYPE_L3_REFINED_COL].nunique(),
}

print("✓ Metadata added")

# Clean up
del adata_hvg, vae, lvae
gc.collect()
print("✓ Memory cleaned")

---
## 9. Define Validation Markers

In [ ]:
print("="*80)
print("9. DEFINING VALIDATION MARKERS")
print("="*80)

VALIDATION_MARKERS = {
    # Alveolar macrophages
    'Resident_Alveolar_Mac': ['MARCO', 'FABP4', 'PPARG', 'SIGLEC1', 'CHIT1', 'MSR1', 'SLC40A1', 'APOE'],
    'Resting_Alveolar_Mac': ['IL10', 'TGFB1', 'MERTK', 'MRC1', 'MSR1', 'APOE', 'GAS6', 'TREM2'],
    
    # Neutrophils
    'Neutrophils': ['FCGR3B', 'CSF3R', 'CXCR2', 'MPO', 'ELANE', 'LCN2', 'S100A8', 'S100A9'],
    
    # Classical monocytes
    'Typical_Classical_Mono': ['FCN1', 'S100A8', 'S100A9', 'LILRB1', 'LGALS3', 'CTSS'],
    'Inflammatory_Classical_Mono': ['IL1B', 'CXCL8', 'PTX3', 'NFKBIA', 'TNF', 'CCL20'],
    
    # cDC2
    'Conventional_cDC2': ['CD1C', 'FCER1A', 'CD1E', 'CLEC10A', 'IRF4', 'CST3', 'HLA-DRA', 'HLA-DPB1'],
    'Langerhans_like_cDC2': ['CD207', 'CD1A', 'CCR6', 'EPCAM', 'LILRB4', 'CXCL14'],
    
    # pDC
    'pDC': ['CLEC4C', 'IL3RA', 'GZMB', 'TCF4', 'IRF7', 'SERPINF1'],
    
    # Mast cells
    'Mast_cells': ['TPSB2', 'TPSAB1', 'CPA3', 'MS4A2', 'KIT', 'HDC', 'GATA2'],
    
    # Interstitial macrophages
    'Inflammatory_Interstitial_Mac': ['CCL20', 'PTX3', 'TIMP1', 'IL1B', 'VEGFA', 'SPP1'],
    'Atypically_activated_Interstitial_Mac': ['CXCL8', 'NFKBIA', 'TIMP1', 'VEGFA', 'SPP1', 'CCL20'],
    'M2_like_Interstitial_Mac': ['C1QA', 'C1QB', 'C1QC', 'STAB1', 'FOLR2', 'MRC1'],
    'CD163L1_Interstitial_Mac': ['CD163L1', 'SELENOP', 'LYVE1', 'FOLR2', 'MRC1', 'C1QA', 'C1QC', 'F13A1', 'CXCL12', 'PLXDC1'],
    'Immunoregulatory_Interstitial_Mac': ['IL10', 'TGFB1', 'MERTK', 'FOLR2', 'CD163L1', 'LYVE1', 'GAS6', 'STAB1'],
    'Low_quality_Interstitial_Mac': ['IL1B', 'CXCL8', 'NFKBIA', 'PTX3', 'TNF', 'MARCKS'],
}

# Create ordered marker list
DOTPLOT_MARKERS_ORDERED = []
for markers in VALIDATION_MARKERS.values():
    DOTPLOT_MARKERS_ORDERED.extend(markers)

# Remove duplicates
seen = set()
DOTPLOT_MARKERS_UNIQUE = []
for marker in DOTPLOT_MARKERS_ORDERED:
    if marker not in seen:
        DOTPLOT_MARKERS_UNIQUE.append(marker)
        seen.add(marker)

print(f"✓ Defined {len(VALIDATION_MARKERS)} marker sets")
print(f"✓ Total unique markers: {len(DOTPLOT_MARKERS_UNIQUE)}")

---
## 10. Generate Comprehensive Dotplot

In [ ]:
print("="*80)
print("10. GENERATING COMPREHENSIVE VALIDATION DOTPLOT")
print("="*80)

# Check availability
available_markers = [m for m in DOTPLOT_MARKERS_UNIQUE if m in adata_full.var_names]
missing_markers = [m for m in DOTPLOT_MARKERS_UNIQUE if m not in adata_full.var_names]

print(f"\nMarker availability:")
print(f"  Available: {len(available_markers)}/{len(DOTPLOT_MARKERS_UNIQUE)} ({len(available_markers)/len(DOTPLOT_MARKERS_UNIQUE)*100:.1f}%)")
if missing_markers:
    print(f"  Missing ({len(missing_markers)}): {', '.join(missing_markers[:20])}")

In [ ]:
# Cell type ordering
celltype_order = [
    'Resident Alveolar macrophages',
    'Resting Alveolar macrophages',
    'Neutrophils',
    'Typical Classical monocytes',
    'Inflammatory Classical monocytes',
    'Conventional cDC2',
    'Langerhans-like cDC2',
    'pDC',
    'Mast cells',
    'Inflammatory Interstitial macrophages',
    'Atypically activated Interstitial macrophages',
    'M2-like Interstitial macrophages',
    'CD163L1+ Interstitial macrophages',
    'Immunoregulatory Interstitial macrophages',
    'Low-quality Interstitial macrophages',
]

celltype_order_filtered = [ct for ct in celltype_order if ct in adata_full.obs[CELLTYPE_L3_REFINED_COL].values]

# Reorder categorical
adata_full.obs[CELLTYPE_L3_REFINED_COL] = pd.Categorical(
    adata_full.obs[CELLTYPE_L3_REFINED_COL],
    categories=celltype_order_filtered,
    ordered=True
)

print(f"\nOrdered {len(celltype_order_filtered)} cell types")

In [ ]:
# Calculate figure size
n_genes = len(available_markers)
n_celltypes = len(celltype_order_filtered)
fig_width = max(30, n_genes * 0.3)
fig_height = max(12, n_celltypes * 0.7)

print(f"\nGenerating dotplot:")
print(f"  Markers: {n_genes}")
print(f"  Cell types: {n_celltypes}")
print(f"  Figure size: {fig_width:.1f} × {fig_height:.1f} inches")

# Generate dotplot
fig, ax = plt.subplots(figsize=(fig_width, fig_height))

sc.pl.dotplot(
    adata_full,
    var_names=available_markers,
    groupby=CELLTYPE_L3_REFINED_COL,
    standard_scale='var',
    show=False,
    ax=ax,
)

fig.suptitle('Myeloid Cell Type Validation - Refined Annotation with Immunoregulatory IM', 
            fontsize=18, y=0.998, weight='bold')
plt.tight_layout()

# Save
output_png = OUTPUT_DIR / 'dotplot_COMPREHENSIVE_VALIDATION_FINAL.png'
output_pdf = OUTPUT_DIR / 'dotplot_COMPREHENSIVE_VALIDATION_FINAL.pdf'

plt.savefig(output_png, dpi=300, bbox_inches='tight')
plt.savefig(output_pdf, bbox_inches='tight')
plt.close()

if output_png.exists() and output_pdf.exists():
    png_size = output_png.stat().st_size / 1024 / 1024
    pdf_size = output_pdf.stat().st_size / 1024 / 1024
    print(f"\n✅ Dotplot saved:")
    print(f"  PNG: {output_png.name} ({png_size:.2f} MB)")
    print(f"  PDF: {output_pdf.name} ({pdf_size:.2f} MB)")
else:
    print(f"\n⚠️  Dotplot files not found")

---
## 11. Generate UMAP Visualizations

In [ ]:
print("="*80)
print("11. GENERATING UMAP VISUALIZATIONS")
print("="*80)

In [ ]:
# UMAP: Refined annotation
fig, ax = plt.subplots(figsize=(12, 10))
sc.pl.umap(adata_full, color=CELLTYPE_L3_REFINED_COL, ax=ax, show=False, legend_loc='right margin')
plt.title('Refined Cell Type Annotation', fontsize=16, weight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'umap_refined_annotation.png', dpi=300, bbox_inches='tight')
plt.savefig(OUTPUT_DIR / 'umap_refined_annotation.pdf', bbox_inches='tight')
plt.close()
print("  ✓ Saved: umap_refined_annotation")

In [ ]:
# UMAP: scANVI predictions
fig, ax = plt.subplots(figsize=(12, 10))
sc.pl.umap(adata_full, color='scanvi_predictions', ax=ax, show=False, legend_loc='right margin')
plt.title('scANVI Predictions', fontsize=16, weight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'umap_scanvi_predictions.png', dpi=300, bbox_inches='tight')
plt.savefig(OUTPUT_DIR / 'umap_scanvi_predictions.pdf', bbox_inches='tight')
plt.close()
print("  ✓ Saved: umap_scanvi_predictions")

In [ ]:
# UMAP: Batch distribution
fig, ax = plt.subplots(figsize=(12, 10))
sc.pl.umap(adata_full, color=BATCH_KEY, ax=ax, show=False, legend_loc='right margin')
plt.title('Batch Distribution', fontsize=16, weight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'umap_batch_distribution.png', dpi=300, bbox_inches='tight')
plt.savefig(OUTPUT_DIR / 'umap_batch_distribution.pdf', bbox_inches='tight')
plt.close()
print("  ✓ Saved: umap_batch_distribution")

In [ ]:
# UMAP: Key markers
key_markers = ['CD163L1', 'MARCO', 'S100A8', 'CD1C', 'TPSAB1', 'IL10']
key_markers_avail = [m for m in key_markers if m in adata_full.var_names]

if key_markers_avail:
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    for i, marker in enumerate(key_markers_avail[:6]):
        sc.pl.umap(adata_full, color=marker, ax=axes[i], show=False, vmax='p99', cmap='Reds')
        axes[i].set_title(marker, fontsize=14, weight='bold')
    
    # Hide unused subplots
    for i in range(len(key_markers_avail), 6):
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'umap_key_markers.png', dpi=300, bbox_inches='tight')
    plt.savefig(OUTPUT_DIR / 'umap_key_markers.pdf', bbox_inches='tight')
    plt.close()
    print("  ✓ Saved: umap_key_markers")
else:
    print("  ⚠️  No key markers available")

---
## 12. Generate Annotation Structure Table

In [ ]:
print("="*80)
print("12. GENERATING ANNOTATION STRUCTURE TABLE")
print("="*80)

annotation_table = []

for original_cluster in sorted(adata_full.obs[CELLTYPE_L3_COL].unique()):
    refined_annotation = CELLTYPE_REANNOTATION.get(original_cluster, original_cluster)
    n_cells = (adata_full.obs[CELLTYPE_L3_COL] == original_cluster).sum()
    pct = n_cells / adata_full.n_obs * 100
    
    l2 = adata_full.obs[adata_full.obs[CELLTYPE_L3_COL] == original_cluster][CELLTYPE_L2_COL].iloc[0]
    is_low_conf = original_cluster in LOW_CONFIDENCE_CLUSTERS
    
    # Check if merged or preserved
    is_merged = False
    merged_note = ''
    if original_cluster in ['Intestinal macrophages_c0', 'Intestinal macrophages_c2', 'Intestinal macrophages_c4']:
        is_merged = True
        merged_note = 'Merged to CD163L1+ IM'
    elif original_cluster == 'DC_c1':
        is_merged = True
        merged_note = 'Merged to Conventional cDC2'
    elif original_cluster == 'Intestinal macrophages_c3':
        merged_note = 'Preserved as Immunoregulatory IM'
    
    annotation_table.append({
        'cluster_id': original_cluster,
        'cell_type_L2': l2,
        'cell_type_L3_refined': refined_annotation,
        'n_cells': n_cells,
        'percent': f"{pct:.2f}%",
        'merged': 'Yes' if is_merged else 'No',
        'notes': merged_note if merged_note else ('Low confidence' if is_low_conf else ''),
    })

df_annotation = pd.DataFrame(annotation_table)

# Save
csv_output = OUTPUT_DIR / 'cell_type_annotation_structure_FINAL.csv'
df_annotation.to_csv(csv_output, index=False)
print(f"\n✓ Saved: {csv_output.name}")

In [ ]:
# Display
print("\n" + "="*80)
print("ANNOTATION STRUCTURE TABLE")
print("="*80)
print(df_annotation.to_string(index=False))

---
## 13. Save Final H5AD

In [ ]:
print("\n" + "="*80)
print("13. SAVING FINAL H5AD")
print("="*80)

output_h5ad = OUTPUT_DIR / 'adata_myeloid_refined_FINAL.h5ad'
adata_full.write_h5ad(output_h5ad, compression='gzip', compression_opts=9)

file_size = output_h5ad.stat().st_size / 1024 / 1024
print(f"\n✓ Saved: {output_h5ad.name} ({file_size:.2f} MB)")

print("\nSaved data structure:")
print(f"  .obs columns:")
print(f"    - {CELLTYPE_L3_COL}: Original L3 annotation")
print(f"    - {CELLTYPE_L3_REFINED_COL}: Refined L3 annotation (PRIMARY)")
print(f"    - scanvi_predictions: scANVI predictions")
print(f"    - scanvi_confidence: Prediction confidence")
print(f"    - leiden_scvi: Leiden clustering")
print(f"  .obsm:")
print(f"    - X_scvi: scVI latent space")
print(f"    - X_scanvi: scANVI latent space")
print(f"    - X_umap: UMAP coordinates (scVI)")
print(f"    - X_umap_scanvi: UMAP coordinates (scANVI)")
print(f"  .var:")
print(f"    - highly_variable: HVG mask")
print(f"  .uns:")
print(f"    - scvi_params: Model hyperparameters")
print(f"    - annotation_info: Annotation metadata")

---
## 14. Summary Statistics

In [ ]:
print("\n" + "="*80)
print("14. SUMMARY STATISTICS")
print("="*80)

print(f"\nFinal dataset:")
print(f"  Total cells: {adata_full.n_obs:,}")
print(f"  Total genes: {adata_full.n_vars:,}")
print(f"  HVGs: {adata_full.var['highly_variable'].sum():,}")
print(f"  Batches: {adata_full.obs[BATCH_KEY].nunique()}")

print(f"\nCell types:")
print(f"  Original L3: {adata_full.obs[CELLTYPE_L3_COL].nunique()}")
print(f"  Refined L3: {adata_full.obs[CELLTYPE_L3_REFINED_COL].nunique()}")

print(f"\nMerging summary:")
print(f"  CD163L1+ IM merged from 3 clusters:")
print(f"    - Intestinal macrophages_c0")
print(f"    - Intestinal macrophages_c2")
print(f"    - Intestinal macrophages_c4")
print(f"  Immunoregulatory IM preserved:")
print(f"    - Intestinal macrophages_c3")
print(f"  Conventional cDC2 merged from 3 clusters:")
print(f"    - DC2_c0, DC_c0, DC_c1")

print(f"\nscANVI prediction accuracy:")
agreement = (adata_full.obs[CELLTYPE_L3_REFINED_COL] == adata_full.obs['scanvi_predictions']).sum()
agreement_pct = agreement / adata_full.n_obs * 100
print(f"  Agreement with refined labels: {agreement_pct:.1f}%")
print(f"  Mean confidence: {adata_full.obs['scanvi_confidence'].mean():.3f}")

---
## 15. Pipeline Complete

In [ ]:
print("\n" + "="*80)
print("✅ PIPELINE COMPLETE")
print("="*80)

print(f"\nGenerated files:")
print(f"  📊 adata_myeloid_refined_FINAL.h5ad")
print(f"  📊 cell_type_annotation_structure_FINAL.csv")
print(f"  📊 dotplot_COMPREHENSIVE_VALIDATION_FINAL.png/pdf")
print(f"  📊 umap_refined_annotation.png/pdf")
print(f"  📊 umap_scanvi_predictions.png/pdf")
print(f"  📊 umap_batch_distribution.png/pdf")
print(f"  📊 umap_key_markers.png/pdf")
print(f"  📂 scvi_model/")
print(f"  📂 scanvi_model/")

print(f"\nOutput directory: {OUTPUT_DIR}")

print("\n" + "="*80)
print("END OF PIPELINE")
print("="*80)